In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.4G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

REGION = variables.TARGET_REGION
ALL_REGIONS = ["nsw", "qld", "vic", "sa"]
OTHER_REGIONS = [r for r in ALL_REGIONS if r != REGION]

# 5-min interval counts for calendar-anchored lags / rolling windows.
PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES   # 12
PER_DAY  = 24 * PER_HOUR                                     # 288
PER_WEEK = 7 * PER_DAY                                       # 2016

In [3]:
df = read_parquet_float32("../1_Dataset/Processed_data/2_dispatch_region_sum.parquet")

df_core_columns = df.columns
df_base = df
df_base[:10]

Loading..: 100%|██████████| 9/9 [00:00<00:00, 29.40batch/s]


,demand_nsw,avail_gen_nsw,interchange_nsw,demand_forecast_nsw,dispatch_gen_nsw,demand_qld,avail_gen_qld,interchange_qld,demand_forecast_qld,dispatch_gen_qld,demand_vic,avail_gen_vic,interchange_vic,demand_forecast_vic,dispatch_gen_vic,demand_sa,avail_gen_sa,interchange_sa,demand_forecast_sa,dispatch_gen_sa
Date,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,7021.709961,11526.263672,-848.010010,-22.460939,6173.700195,6057.979980,10651.650391,823.799988,-61.518551,6881.779785,4306.370117,8726.033203,600.659973,-12.512210,4907.029785,1311.689941,2097.706055,-341.820007,-10.16947,969.880005
2018-01-01 00:10:00,6952.049805,11531.112305,-819.469971,-29.104000,6132.580078,6131.600098,10651.000000,839.669983,-36.184078,6971.270020,4321.689941,8715.471680,574.780029,-11.244630,4896.470215,1300.790039,2086.089111,-378.630005,-12.92362,922.159973
2018-01-01 00:15:00,6950.970215,11519.082031,-878.929993,-29.913090,6072.040039,6050.430176,10651.000000,873.609985,-36.333500,6924.029785,4254.620117,8720.274414,646.659973,-26.593260,4901.270020,1288.660034,2073.114990,-416.549988,-6.01374,872.119995
2018-01-01 00:20:00,6889.879883,11424.864258,-919.890015,-41.521000,5969.990234,6016.680176,10641.000000,860.869995,-38.204590,6877.549805,4233.020020,8718.517578,666.500000,-16.410641,4899.520020,1269.000000,2071.777100,-368.220001,-9.48042,900.780029
2018-01-01 00:25:00,6846.540039,11424.723633,-898.830017,-36.577148,5947.709961,5993.089844,10641.000000,884.510010,-40.438480,6877.600098,4201.049805,8711.393555,691.349976,-19.390631,4892.390137,1289.670044,2065.736084,-424.940002,-9.63593,864.739990
2018-01-01 00:30:00,6812.330078,11433.789062,-932.390015,-44.602539,5879.939941,5935.270020,10641.000000,910.479980,-90.682129,6845.750000,4190.299805,8704.847656,695.549988,-21.152830,4885.850098,1274.810059,2071.377930,-426.779999,-9.40791,848.030029
2018-01-01 00:35:00,6839.930176,11433.101562,-1020.559998,-29.209961,5819.370117,5965.209961,10641.000000,896.140015,-50.729488,6861.350098,4152.569824,8715.330078,743.760010,-21.379881,4896.330078,1231.609985,2068.435059,-372.899994,-11.98604,858.710022
2018-01-01 00:40:00,6841.149902,11434.149414,-1033.650024,-21.369631,5807.500000,5975.629883,10641.000000,881.780029,-18.714840,6857.419922,4121.669922,8727.507812,786.840027,-16.919430,4908.509766,1258.380005,2061.987061,-399.390015,-9.60396,858.989990
2018-01-01 00:45:00,6794.399902,11436.178711,-1067.010010,-29.372070,5727.390137,5908.029785,10641.000000,932.619995,-44.557621,6840.649902,4090.229980,8741.557617,832.320007,-16.879881,4922.560059,1228.380005,2039.072021,-421.329987,-9.08932,807.049988


In [4]:
def _add_supply_balance_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Instantaneous supply-tightness signals for the target region from
    DISPATCHREGIONSUM. Reserve margin, AEMO's dispatch demand-forecast error
    and thermal headroom are leading indicators of scarcity pricing. Every
    input is known at (or before) the 5-min interval, so there is no leakage.
    Returns only the new columns to avoid copying the full base frame.
    """
    dm = df[f"demand_{REGION}"]
    ag = df[f"avail_gen_{REGION}"]
    dc = df[f"demand_forecast_{REGION}"]
    ic = df[f"interchange_{REGION}"]

    new_cols = {}
    new_cols[f"reserve_margin_{REGION}"]      = ((ag - dm) / (dm + 1)).clip(-2, 10).astype(np.float32)
    new_cols[f"reserve_headroom_mw_{REGION}"] = (ag - dm).clip(-5000, 20000).astype(np.float32)
    new_cols[f"demand_fcst_error_{REGION}"]   = (dc - dm).astype(np.float32)
    new_cols[f"import_dependence_{REGION}"]   = (ic / (dm + 1)).clip(-1, 1).astype(np.float32)

    if f"dispatch_gen_{REGION}" in df.columns:
        dg = df[f"dispatch_gen_{REGION}"]
        new_cols[f"thermal_util_{REGION}"]       = (dg / (ag + 1)).clip(0, 1.5).astype(np.float32)
        new_cols[f"thermal_surplus_mw_{REGION}"] = (ag - dg).clip(-2000, 15000).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_supply_balance_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,reserve_margin_nsw,reserve_headroom_mw_nsw,demand_fcst_error_nsw,import_dependence_nsw,thermal_util_nsw,thermal_surplus_mw_nsw
Date,,,,,,
2018-01-01 00:05:00,0.641427,4504.553711,-7044.170898,-0.120753,0.535574,5352.563477
2018-01-01 00:10:00,0.658569,4579.062500,-6981.153809,-0.117858,0.531783,5398.532227
2018-01-01 00:15:00,0.657096,4568.111816,-6980.883301,-0.126429,0.527083,5447.041992
2018-01-01 00:20:00,0.658114,4534.984375,-6931.400879,-0.133494,0.522498,5454.874023
2018-01-01 00:25:00,0.668588,4578.183594,-6883.117188,-0.131263,0.520554,5477.013672
2018-01-01 00:30:00,0.678297,4621.458984,-6856.932617,-0.136848,0.514215,5553.849121
2018-01-01 00:35:00,0.671425,4593.171387,-6869.140137,-0.149184,0.508949,5613.731445
2018-01-01 00:40:00,0.671280,4592.999512,-6862.519531,-0.151071,0.507864,5626.649414
2018-01-01 00:45:00,0.683077,4641.778809,-6823.771973,-0.157019,0.500769,5708.788574


In [5]:
def _add_region_lag_roll_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Backward-looking lags and rolling statistics for each core system series of
    the target region. Calendar-anchored lags (1h / 1d / 2d / 1w) capture the
    strong daily and weekly demand seasonality. Look-back only, no leakage.
    Returns only the new columns to avoid copying the full base frame.
    """
    series = {
        "demand":          df[f"demand_{REGION}"],
        "avail_gen":       df[f"avail_gen_{REGION}"],
        "interchange":     df[f"interchange_{REGION}"],
        "demand_forecast": df[f"demand_forecast_{REGION}"],
    }
    if f"dispatch_gen_{REGION}" in df.columns:
        series["dispatch_gen"] = df[f"dispatch_gen_{REGION}"]

    LAGS = [1, 2, 3, PER_HOUR, 3 * PER_HOUR, PER_DAY, 2 * PER_DAY, PER_WEEK]
    new_cols = {}
    for name, s in series.items():
        for lag in LAGS:
            new_cols[f"{name}_{REGION}_lag_{lag}"] = s.shift(lag).astype(np.float32)
        for w in [PER_HOUR, 4 * PER_HOUR, PER_DAY]:
            r = s.rolling(w, min_periods=max(1, w // 2))
            new_cols[f"{name}_{REGION}_rmean_{w}"] = r.mean().astype(np.float32)
            new_cols[f"{name}_{REGION}_rstd_{w}"]  = r.std().astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_region_lag_roll_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,demand_nsw_lag_1,demand_nsw_lag_2,demand_nsw_lag_3,demand_nsw_lag_12,demand_nsw_lag_36,demand_nsw_lag_288,demand_nsw_lag_576,demand_nsw_lag_2016,demand_nsw_rmean_12,demand_nsw_rstd_12,...,dispatch_gen_nsw_lag_36,dispatch_gen_nsw_lag_288,dispatch_gen_nsw_lag_576,dispatch_gen_nsw_lag_2016,dispatch_gen_nsw_rmean_12,dispatch_gen_nsw_rstd_12,dispatch_gen_nsw_rmean_48,dispatch_gen_nsw_rstd_48,dispatch_gen_nsw_rmean_288,dispatch_gen_nsw_rstd_288
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,7021.709961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,6952.049805,7021.709961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,6950.970215,6952.049805,7021.709961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,6889.879883,6950.970215,6952.049805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,6846.540039,6889.879883,6950.970215,NaN,NaN,NaN,NaN,NaN,6912.246582,77.287743,...,NaN,NaN,NaN,NaN,6029.326660,114.741371,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,6812.330078,6846.540039,6889.879883,NaN,NaN,NaN,NaN,NaN,6901.915527,75.663239,...,NaN,NaN,NaN,NaN,5999.333008,131.410492,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,6839.930176,6812.330078,6846.540039,NaN,NaN,NaN,NaN,NaN,6894.319824,73.271004,...,NaN,NaN,NaN,NaN,5975.354004,139.290131,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,6841.149902,6839.930176,6812.330078,NaN,NaN,NaN,NaN,NaN,6883.217773,76.202995,...,NaN,NaN,NaN,NaN,5947.802246,154.299377,NaN,NaN,NaN,NaN


In [6]:
def _add_demand_dynamics_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Demand momentum, ramps and level context for the target region. Fast ramps
    and demand sitting near recent highs precede scarcity pricing; same-period
    lags (yesterday / last year) exploit seasonality. Backward-looking only.
    Returns only the new columns to avoid copying the full base frame.
    """
    dm = df[f"demand_{REGION}"]
    new_cols = {}

    new_cols[f"demand_mom_{REGION}_1h"]   = dm.diff(PER_HOUR).astype(np.float32)
    new_cols[f"demand_mom_{REGION}_3h"]   = dm.diff(3 * PER_HOUR).astype(np.float32)
    new_cols[f"demand_ramp_{REGION}_30m"] = dm.diff(6).astype(np.float32)

    new_cols[f"demand_rmax_{REGION}_1d"] = dm.rolling(PER_DAY, min_periods=PER_HOUR).max().astype(np.float32)
    new_cols[f"demand_rmin_{REGION}_1d"] = dm.rolling(PER_DAY, min_periods=PER_HOUR).min().astype(np.float32)
    new_cols[f"demand_pct_rank_{REGION}_1w"] = (
        dm.rolling(PER_WEEK, min_periods=PER_DAY).rank(pct=True).astype(np.float32)
    )

    ANNUAL = int(round(365.25 * PER_DAY))
    new_cols[f"demand_lag_{REGION}_annual"] = dm.shift(ANNUAL).astype(np.float32)
    new_cols[f"demand_yoy_change_{REGION}"] = (dm - dm.shift(ANNUAL)).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_demand_dynamics_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,demand_mom_nsw_1h,demand_mom_nsw_3h,demand_ramp_nsw_30m,demand_rmax_nsw_1d,demand_rmin_nsw_1d,demand_pct_rank_nsw_1w,demand_lag_nsw_annual,demand_yoy_change_nsw
Date,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,NaN,NaN,-181.779785,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,NaN,NaN,-110.899902,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,NaN,NaN,-156.570312,NaN,NaN,NaN,NaN,NaN


In [7]:
def _add_cross_region_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    NEM-wide and neighbouring-region context. Tight supply or high demand in an
    adjacent region raises interconnector import prices into the target region.
    Uses only contemporaneous / past values, so it is leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}

    demand_cols = [f"demand_{r}"    for r in ALL_REGIONS if f"demand_{r}"    in df.columns]
    avail_cols  = [f"avail_gen_{r}" for r in ALL_REGIONS if f"avail_gen_{r}" in df.columns]
    nem_demand = df[demand_cols].sum(axis=1)
    nem_avail  = df[avail_cols].sum(axis=1)

    new_cols["nem_demand_total"]   = nem_demand.astype(np.float32)
    new_cols["nem_avail_total"]    = nem_avail.astype(np.float32)
    new_cols["nem_reserve_margin"] = ((nem_avail - nem_demand) / (nem_demand + 1)).clip(-2, 10).astype(np.float32)
    new_cols[f"demand_share_{REGION}"] = (df[f"demand_{REGION}"] / (nem_demand + 1)).clip(0, 1).astype(np.float32)

    for r in OTHER_REGIONS:
        if f"demand_{r}" in df.columns:
            new_cols[f"neighbour_demand_{r}"] = df[f"demand_{r}"].astype(np.float32)
        if f"interchange_{r}" in df.columns:
            new_cols[f"neighbour_interchange_{r}"] = df[f"interchange_{r}"].astype(np.float32)
        if f"avail_gen_{r}" in df.columns and f"demand_{r}" in df.columns:
            new_cols[f"neighbour_reserve_margin_{r}"] = (
                (df[f"avail_gen_{r}"] - df[f"demand_{r}"]) / (df[f"demand_{r}"] + 1)
            ).clip(-2, 10).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_cross_region_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,nem_demand_total,nem_avail_total,nem_reserve_margin,demand_share_nsw,neighbour_demand_qld,neighbour_interchange_qld,neighbour_reserve_margin_qld,neighbour_demand_vic,neighbour_interchange_vic,neighbour_reserve_margin_vic,neighbour_demand_sa,neighbour_interchange_sa,neighbour_reserve_margin_sa
Date,,,,,,,,,,,,,
2018-01-01 00:05:00,18697.748047,33001.652344,0.764966,0.375518,6057.979980,823.799988,0.758159,4306.370117,600.659973,1.026070,1311.689941,-341.820007,0.598783
2018-01-01 00:10:00,18706.128906,32983.675781,0.763214,0.371626,6131.600098,839.669983,0.736947,4321.689941,574.780029,1.016446,1300.790039,-378.630005,0.603246
2018-01-01 00:15:00,18544.679688,32963.468750,0.777474,0.374803,6050.430176,873.609985,0.760245,4254.620117,646.659973,1.049355,1288.660034,-416.549988,0.608265
2018-01-01 00:20:00,18408.580078,32856.156250,0.784786,0.374255,6016.680176,860.869995,0.768456,4233.020020,666.500000,1.059394,1269.000000,-368.220001,0.632108
2018-01-01 00:25:00,18330.349609,32842.851562,0.791677,0.373488,5993.089844,884.510010,0.775415,4201.049805,691.349976,1.073367,1289.670044,-424.940002,0.601289
2018-01-01 00:30:00,18212.708984,32851.015625,0.803697,0.374022,5935.270020,910.479980,0.792708,4190.299805,695.549988,1.077124,1274.810059,-426.779999,0.624362
2018-01-01 00:35:00,18189.320312,32857.867188,0.806393,0.376020,5965.209961,896.140015,0.783712,4152.569824,743.760010,1.098515,1231.609985,-372.899994,0.678905
2018-01-01 00:40:00,18196.830078,32864.644531,0.806020,0.375932,5975.629883,881.780029,0.780602,4121.669922,786.840027,1.117198,1258.380005,-399.390015,0.638097
2018-01-01 00:45:00,18021.041016,32857.808594,0.823257,0.377005,5908.029785,932.619995,0.800972,4090.229980,832.320007,1.136902,1228.380005,-421.329987,0.659432


In [8]:
# Retain core columns: contemporaneous system state (demand, availability,
# interchange, dispatch demand-forecast) is known at origin t -> leakage-free.
print("Total features:", df.shape[1])
df.to_parquet("../2_Features_build/Feature_data/2_dispatch_region_sum.parquet")
df.shape

Total features: 117


(893664, 117)

In [9]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 15 variable(s); kernel rss 0.62G, 8.4G RAM free now
